# Base Modeling

Este notebook parte del **dataset final** de Data Preparation (`dataset_final.parquet`, guardado en la última celda de `DataPreparation.ipynb`).

Aquí se hacen la normalización de los datos y la definición de los folds de validación con `TimeSeriesSplit`.

### 1. Conectar con Google Drive e importar librerías

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from google.colab import drive

drive.mount('/content/drive')

### 2. Cargar el dataset final
Se lee desde `Datos/Procesado`. Si no existe, primero hay que ejecutar `DataPreparation.ipynb` completo.

In [ ]:
RUTA_DATOS = "/content/drive/My Drive/Datos/Procesado"

dataset_final = pd.read_parquet(os.path.join(RUTA_DATOS, "dataset_final.parquet"))
dataset_final["bloque_tiempo"] = pd.to_datetime(dataset_final["bloque_tiempo"])
dataset_final = dataset_final.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

print(f"dataset_final: {len(dataset_final):,} bloques x {dataset_final.shape[1]} columnas")
print(list(dataset_final.columns))

### 3. Parámetros

In [ ]:
TARGET = "pct_posible_cola"
N_FOLDS = 5
tscv = TimeSeriesSplit(n_splits=N_FOLDS)

# Normalización de datos

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pandas as pd

TARGET = "pct_posible_cola"

df_model = dataset_final.copy()

columnas_excluir = [TARGET, "ms", "bloque_tiempo"]

X = df_model.drop(columns=columnas_excluir)
y = df_model[TARGET]

categoricas = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numericas = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categoricas),
    ],
    remainder="drop",
    verbose_feature_names_out=False,   # quita los prefijos num__ / cat__
)

# Que devuelva un DataFrame directamente (sklearn >= 1.2)
preprocessor.set_output(transform="pandas")

X_normalizado = preprocessor.fit_transform(X)

In [ ]:
no_normalizadas = df_model[columnas_excluir]   # TARGET, ms, bloque_tiempo

# Reintegrar por índice
df_final = pd.concat([no_normalizadas, X_normalizado], axis=1)

> **Nota:** en esta celda el `preprocessor` se ajusta con todo el dataset, lo que deja ver información del periodo de prueba. Para entrenar y evaluar modelos se debe ajustar (`fit`) solo con los bloques de entrenamiento de cada fold y aplicar `transform` a los de prueba.

# Validación

Se usa el mismo `TimeSeriesSplit` de Data Preparation (5 folds), ahora sobre `dataset_final`: cada fold entrena con el pasado y prueba con el periodo siguiente.

In [ ]:
df_final = df_final.sort_values(["bloque_tiempo", "ms"]).reset_index(drop=True)

folds = []
for i, (idx_ent, idx_prueba) in enumerate(tscv.split(df_final), start=1):
    t = df_final["bloque_tiempo"]
    folds.append({
        "fold": i,
        "entrenamiento": f"{t.iloc[idx_ent[0]].date()} a {t.iloc[idx_ent[-1]].date()}",
        "prueba": f"{t.iloc[idx_prueba[0]].date()} a {t.iloc[idx_prueba[-1]].date()}",
        "bloques_entrenamiento": len(idx_ent),
        "bloques_prueba": len(idx_prueba),
    })
display(pd.DataFrame(folds))
